# Prompt Caching for Contextual RAG
## Teaching Notebook 2026

This notebook teaches **two ideas**:

1. **Prompt Caching** - how LLM providers reuse computed KV-caches to cut costs 50-90%
2. **Contextual RAG** - how prompt caching powers cheap chunk contextualization

| Section | Topic |
|---------|-------|
| 1 | Setup - three provider clients |
| 2 | Document loading and chunking |
| 3 | Theory - what prompt caching is |
| 4 | Anthropic - Explicit caching with `cache_control` |
| 5 | OpenAI - Implicit automatic prefix caching |
| 6 | DeepSeek via OpenRouter - Implicit, cheapest option |
| 7 | Cost comparison across all three providers |
| 8 | Full RAG pipeline - contextual embeddings + cached LLM calls |

**APIs Required**: ANTHROPIC_API_KEY, OPENAI_API_KEY, OPENROUTER_API_KEY, VOYAGE_API_KEY

## 1. Setup

In [ ]:
#!uv pip install -q anthropic openai voyageai python-dotenv matplotlib tqdm numpy pypdf
#print("Packages ready")

In [1]:
import os, json, pickle, time, threading
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import anthropic
import voyageai
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY  = os.getenv('ANTHROPIC_API_KEY')
OPENAI_API_KEY     = os.getenv('OPENAI_API_KEY')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
VOYAGE_API_KEY     = os.getenv('VOYAGE_API_KEY')

for name, val in [
    ('ANTHROPIC_API_KEY',  ANTHROPIC_API_KEY),
    ('OPENAI_API_KEY',     OPENAI_API_KEY),
    ('OPENROUTER_API_KEY', OPENROUTER_API_KEY),
    ('VOYAGE_API_KEY',     VOYAGE_API_KEY),
]:
    ok = val is not None
    print(f"{'OK' if ok else 'MISSING'}: {name}")

print("\nImports complete!")

OK: ANTHROPIC_API_KEY
OK: OPENAI_API_KEY
OK: OPENROUTER_API_KEY
OK: VOYAGE_API_KEY

Imports complete!


In [2]:
anthropic_client  = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client     = OpenAI(api_key=OPENAI_API_KEY)
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)

ANTHROPIC_MODEL = 'claude-sonnet-4-6'
OPENAI_MODEL    = 'gpt-4o-mini'
DEEPSEEK_MODEL  = 'deepseek/deepseek-chat'
EMBEDDING_MODEL = 'voyage-2'
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 200
DEFAULT_K       = 5

PRICING = {
    'anthropic': {'input': 0.80, 'cache_write': 1.00, 'cache_read': 0.08,  'output': 4.00},
    'openai':    {'input': 0.15, 'cache_write': 0.15, 'cache_read': 0.075, 'output': 0.60},
    'deepseek':  {'input': 0.14, 'cache_write': 0.14, 'cache_read': 0.014, 'output': 0.28},
}

print("All clients initialized!")
print(f"  Anthropic (Explicit): {ANTHROPIC_MODEL}")
print(f"  OpenAI    (Implicit): {OPENAI_MODEL}")
print(f"  DeepSeek  (Implicit): {DEEPSEEK_MODEL} via OpenRouter")

All clients initialized!
  Anthropic (Explicit): claude-sonnet-4-6
  OpenAI    (Implicit): gpt-4o-mini
  DeepSeek  (Implicit): deepseek/deepseek-chat via OpenRouter


## 2. Load & Chunk Documents

Uses the same `data/documents/` folder as Part 1 (the two Markdown files = 27 chunks).

In [3]:
def load_documents_from_folder(folder_path: str) -> List[Dict[str, Any]]:
    folder = Path(folder_path)
    if not folder.exists():
        folder.mkdir(parents=True, exist_ok=True)
        return []
    files = [f for f in folder.iterdir() if f.suffix.lower() in {'.pdf', '.md', '.txt'}]
    if not files:
        print(f'No documents in {folder_path}')
        return []
    documents = []
    for fp in tqdm(files, desc='Loading'):
        try:
            if fp.suffix.lower() == '.pdf':
                from pypdf import PdfReader
                content = '\n'.join(p.extract_text() for p in PdfReader(fp).pages)
            else:
                content = fp.read_text(encoding='utf-8')
            documents.append({'doc_id': fp.stem, 'content': content.strip(),
                               'filename': fp.name, 'chunks': []})
            print(f'  Loaded: {fp.name} ({len(content):,} chars)')
        except Exception as e:
            print(f'  Error: {fp.name}: {e}')
    return documents


def chunk_documents(docs, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    total = 0
    for doc in docs:
        doc['chunks'] = []
        for i in range(0, len(doc['content']), size - overlap):
            text = doc['content'][i : i + size].strip()
            if text:
                doc['chunks'].append({
                    'chunk_id': f"{doc['doc_id']}_chunk_{len(doc['chunks'])}",
                    'original_index': len(doc['chunks']),
                    'content': text,
                })
        total += len(doc['chunks'])
    print(f'{total} chunks from {len(docs)} documents')
    return docs


documents = load_documents_from_folder('data/documents')
if documents:
    documents   = chunk_documents(documents)
    total_chunks = sum(len(d['chunks']) for d in documents)
    first_doc   = documents[0]
    demo_chunks  = first_doc['chunks'][:5]
    print(f"\nReady: {len(documents)} docs, {total_chunks} chunks")
    print(f"Demo will use first {len(demo_chunks)} chunks of '{first_doc['filename']}'")

Loading: 100%|██████████| 2/2 [00:00<00:00, 1986.41it/s]

  Loaded: neural_networks.md (8,436 chars)
  Loaded: machine_learning_intro.md (6,768 chars)
27 chunks from 2 documents

Ready: 2 docs, 27 chunks
Demo will use first 5 chunks of 'neural_networks.md'


## 3. Theory - What Is Prompt Caching?

### The Expensive Step

Every time an LLM processes a prompt, it builds a **KV (Key-Value) attention cache** for each token.
This computation is the most expensive part of inference.

**Prompt caching** saves that computed KV cache and reuses it across requests that share the same prefix.

```
WITHOUT caching:
  Call 1: [Document 2000 tokens] + [Chunk A 150 tokens]  ->  process 2150 tokens
  Call 2: [Document 2000 tokens] + [Chunk B 150 tokens]  ->  process 2150 tokens  <- full cost again!
  Call 27: same ...

WITH caching:
  Call 1:  [Document 2000 tokens] + [Chunk A 150 tokens]  ->  process 2150, SAVE KV cache
  Call 2:  [Document 2000 tokens] + [Chunk B 150 tokens]  ->  LOAD cached, process 150 only  <- 93% cheaper!
  Call 27: same ...
```

**Result: 2-10x faster Time-To-First-Token + 50-90% cheaper for cached tokens.**

---

### The Golden Rule

> **Static content first. Dynamic content last. Always.**

The cache requires an **exact prefix match**. Dynamic content before static = no cache hit.

```
CORRECT:  system=[document]  user=[chunk query]   <- document cached, chunk not
WRONG:    user=[chunk query + document]            <- prefix changes every call, no cache hit
```

---

### Explicit vs Implicit Caching

| Provider | Type | Activation | Min tokens | Discount |
|----------|------|------------|------------|----------|
| Anthropic | Explicit | Add `cache_control: ephemeral` to content block | ~200 | 90% off |
| OpenAI | Implicit | Nothing - automatic | >=1024 | 50% off |
| DeepSeek | Implicit | Nothing - automatic | >=64 | 90% off |

## 4. Provider 1 - Anthropic Explicit Caching

Anthropic requires you to **explicitly mark** which content blocks to cache:

```python
{'type': 'text', 'text': '<document>...</document>', 'cache_control': {'type': 'ephemeral'}}
```

- `ephemeral` = 5-minute TTL
- Cache write: 1.25x normal input price (one-time)
- Cache read: 0.10x normal input price (90% off!)
- Fields: `usage.cache_creation.ephemeral_5m_input_tokens` (write) and `usage.cache_read_input_tokens` (read)
- **Note**: Claude 4.x SDK v0.76+ stores write tokens in a nested `cache_creation` object, not top-level `cache_creation_input_tokens`

In [4]:
# ✅ AnthropicCachingLLM v2 — uses system param + nested cc_obj (Claude 4.x SDK fix)
class AnthropicCachingLLM:
    # Explicit caching via system parameter — the ONLY reliable path for cache_control.
    # cache_control in user content blocks is silently ignored by the Anthropic API.
    # Document goes in system=[{...cache_control...}], chunk stays in user message.

    _CHUNK_PROMPT = (
        'Here is the chunk to situate within the document above:\n'
        '<chunk>\n{chunk}\n</chunk>\n\n'
        'Give a short succinct context to situate this chunk for search retrieval. Nothing else.'
    )

    def situate_context(self, doc: str, chunk: str) -> Tuple[str, dict]:
        response = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=500,
            system=[
                {
                    'type': 'text',
                    'text': f'<document>\n{doc}\n</document>',
                    'cache_control': {'type': 'ephemeral'},  # <- CACHED: same doc reused across chunks
                }
            ],
            messages=[{
                'role': 'user',
                'content': self._CHUNK_PROMPT.format(chunk=chunk),  # <- NOT cached: changes per chunk
            }],
        )
        # Claude 4.x SDK stores write tokens in usage.cache_creation.ephemeral_5m_input_tokens
        # (nested object), NOT in usage.cache_creation_input_tokens (always 0 for Claude 4.x)
        cc_obj = getattr(response.usage, 'cache_creation', None)
        cc5m   = getattr(cc_obj, 'ephemeral_5m_input_tokens', 0) or 0
        cr     = getattr(response.usage, 'cache_read_input_tokens', 0) or 0
        # DEBUG: uncomment next line if still seeing UNKNOWN
        # print(f'  [debug] input={response.usage.input_tokens} cc5m={cc5m} cr={cr} raw_usage={response.usage.__dict__}')
        return response.content[0].text, {
            'input':          response.usage.input_tokens,
            'output':         response.usage.output_tokens,
            'cache_creation': cc5m,
            'cache_read':     cr,
        }

anthropic_llm = AnthropicCachingLLM()
print('✅ AnthropicCachingLLM v2 loaded — system param + cc_obj fix active')

✅ AnthropicCachingLLM v2 loaded — system param + cc_obj fix active


### Demo - Watch Cache Hits Appear

Call 1 = cache MISS (document written to cache).  
Calls 2-5 = cache HIT (reads at 90% off).

In [5]:
if not documents:
    print('No documents found.')
else:
    print(f"Document: {first_doc['filename']} ({len(first_doc['content']):,} chars)")
    print(f"Running on {len(demo_chunks)} chunks\n")
    print(f"{'Call':<6} {'cache_creation':>16} {'cache_read':>12}  Status")
    print('-' * 55)

    anthropic_totals   = {'input': 0, 'output': 0, 'cache_creation': 0, 'cache_read': 0}
    anthropic_contexts = []

    for i, chunk in enumerate(demo_chunks, 1):
        ctx, usage = anthropic_llm.situate_context(first_doc['content'], chunk['content'])
        anthropic_contexts.append(ctx)
        for k in anthropic_totals:
            anthropic_totals[k] += usage[k]
        cc, cr = usage['cache_creation'], usage['cache_read']
        if cr > 0 and cc > 0:
            status = 'HIT + new write (doc cached, chunk cached)'
        elif cr > 0:
            status = 'HIT  – reading from cache'
        elif cc > 0:
            status = 'MISS – writing to cache'
        else:
            status = 'UNKNOWN – no cache stats'
        print(f"  {i:<4} {usage['cache_creation']:>16,} {usage['cache_read']:>12,}  {status}")

    print(f"\nTotals:")
    for k, v in anthropic_totals.items():
        print(f"   {k:<20} {v:>10,}")
    print(f"\nContext for chunk 1: {anthropic_contexts[0][:200]}")

Document: neural_networks.md (8,435 chars)
Running on 5 chunks

Call     cache_creation   cache_read  Status
-------------------------------------------------------
  1               2,203            0  MISS – writing to cache
  2                   0        2,203  HIT  – reading from cache
  3                   0        2,203  HIT  – reading from cache
  4                   0        2,203  HIT  – reading from cache
  5                   0        2,203  HIT  – reading from cache

Totals:
   input                     1,348
   output                      285
   cache_creation            2,203
   cache_read                8,812

Context for chunk 1: This chunk is the beginning of the document, covering the introduction/overview and the start of the Neural Network Architecture section, specifically introducing basic layer structure and beginning t


In [6]:
p  = PRICING['anthropic']
t  = anthropic_totals
n  = len(demo_chunks)

cost_with = (
    t['input']          * p['input']       / 1_000_000
    + t['cache_creation'] * p['cache_write'] / 1_000_000
    + t['cache_read']     * p['cache_read']  / 1_000_000
    + t['output']         * p['output']      / 1_000_000
)
all_input    = t['input'] + t['cache_creation'] + t['cache_read']
cost_without = all_input * p['input'] / 1_000_000 + t['output'] * p['output'] / 1_000_000
savings      = (1 - cost_with / cost_without) * 100 if cost_without > 0 else 0

print(f'Anthropic Cost Analysis ({n} chunks)')
print(f'  Cost WITH caching:    ${cost_with:.6f}')
print(f'  Cost WITHOUT caching: ${cost_without:.6f}')
print(f'  Savings:              {savings:.1f}%')
print(f"\nExtrapolated to {total_chunks} chunks:")
scale = total_chunks / n
print(f'  WITH:    ${cost_with * scale:.4f}')
print(f'  WITHOUT: ${cost_without * scale:.4f}')

Anthropic Cost Analysis (5 chunks)
  Cost WITH caching:    $0.005126
  Cost WITHOUT caching: $0.011030
  Savings:              53.5%

Extrapolated to 27 chunks:
  WITH:    $0.0277
  WITHOUT: $0.0596


## 5. Provider 2 - OpenAI Implicit Caching

No headers needed. OpenAI caches automatically when prefix >= 1024 tokens.

- Cache reads cost 50% of normal input price
- Track via `response.usage.prompt_tokens_details.cached_tokens`

Put document in **system message** (static prefix), chunk in **user message** (dynamic).

In [7]:
class OpenAICachingLLM:
    # Implicit caching: no headers, just structure the prompt correctly

    _CHUNK_PROMPT = (
        'Here is the chunk to situate within the document above:\n'
        '<chunk>\n{chunk}\n</chunk>\n\n'
        'Give a short succinct context to situate this chunk for search retrieval. Nothing else.'
    )

    def situate_context(self, doc: str, chunk: str) -> Tuple[str, dict]:
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {
                    'role': 'system',                       # <- STATIC prefix (auto-cached >=1024 tokens)
                    'content': f'<document>\n{doc}\n</document>',
                },
                {
                    'role': 'user',                         # <- DYNAMIC suffix
                    'content': self._CHUNK_PROMPT.format(chunk=chunk),
                },
            ],
        )
        details = response.usage.prompt_tokens_details
        cached  = getattr(details, 'cached_tokens', 0) if details else 0
        return response.choices[0].message.content, {
            'input':          response.usage.prompt_tokens,
            'output':         response.usage.completion_tokens,
            'cache_creation': 0,
            'cache_read':     cached,
        }

openai_llm = OpenAICachingLLM()
print("OpenAICachingLLM ready")

OpenAICachingLLM ready


In [8]:
if not documents:
    print('No documents.')
else:
    print(f"{'Call':<6} {'total_input':>12} {'cached_tokens':>14}  Status")
    print('-' * 50)

    openai_totals   = {'input': 0, 'output': 0, 'cache_creation': 0, 'cache_read': 0}
    openai_contexts = []

    for i, chunk in enumerate(demo_chunks, 1):
        ctx, usage = openai_llm.situate_context(first_doc['content'], chunk['content'])
        openai_contexts.append(ctx)
        for k in openai_totals:
            openai_totals[k] += usage[k]
        status = 'HIT' if usage['cache_read'] > 0 else 'MISS'
        print(f"  {i:<4} {usage['input']:>12,} {usage['cache_read']:>14,}  {status}")

    print(f"\nTotals:")
    for k, v in openai_totals.items():
        print(f"   {k:<20} {v:>10,}")

Call    total_input  cached_tokens  Status
--------------------------------------------------
  1           2,087              0  MISS
  2           2,156          1,792  HIT
  3           2,145          1,792  HIT
  4           2,153          1,792  HIT
  5           2,136          1,792  HIT

Totals:
   input                    10,677
   output                      256
   cache_creation                0
   cache_read                7,168


In [9]:
p  = PRICING['openai']
t  = openai_totals
n  = len(demo_chunks)

cost_with    = ((t['input'] - t['cache_read']) * p['input'] / 1_000_000
                + t['cache_read'] * p['cache_read'] / 1_000_000
                + t['output'] * p['output'] / 1_000_000)
cost_without = t['input'] * p['input'] / 1_000_000 + t['output'] * p['output'] / 1_000_000
savings      = (1 - cost_with / cost_without) * 100 if cost_without > 0 else 0

print(f'OpenAI Cost Analysis ({n} chunks)')
print(f'  Cost WITH caching:    ${cost_with:.6f}')
print(f'  Cost WITHOUT caching: ${cost_without:.6f}')
print(f'  Savings:              {savings:.1f}%')
scale = total_chunks / n
print(f"\nExtrapolated to {total_chunks} chunks:")
print(f'  WITH:    ${cost_with * scale:.4f}')
print(f'  WITHOUT: ${cost_without * scale:.4f}')

OpenAI Cost Analysis (5 chunks)
  Cost WITH caching:    $0.001218
  Cost WITHOUT caching: $0.001755
  Savings:              30.6%

Extrapolated to 27 chunks:
  WITH:    $0.0066
  WITHOUT: $0.0095


## 6. Provider 3 - DeepSeek via OpenRouter

### How It Works

DeepSeek V3 caches automatically. Accessed via OpenRouter using the standard OpenAI SDK.

- Minimum prefix: **64 tokens** (any real document qualifies)
- Cache discount: **90% off** upstream
- OpenRouter **normalizes** all provider responses to OpenAI format

### Important: Field Names on OpenRouter

OpenRouter wraps all providers in OpenAI-compatible format. Cache stats for DeepSeek come through the **same field as OpenAI** — NOT DeepSeek's native field names:

```python
# CORRECT on OpenRouter (OpenAI-compatible format)
details = response.usage.prompt_tokens_details
cached  = getattr(details, 'cached_tokens', 0)      # same as OpenAI

# WRONG — these are DeepSeek's native API fields, not available on OpenRouter
response.usage.prompt_cache_read_tokens   # doesn't exist on OpenRouter
response.usage.prompt_cache_write_tokens  # doesn't exist on OpenRouter
```

### Caveat

OpenRouter may not always populate `cached_tokens` even when caching is active.
Check the `response.usage.cost` field — if it drops across calls for the same document, caching IS happening upstream even if stats show 0.

**Alternative model**: `qwen/qwen-2.5-72b-instruct` (also implicit caching via OpenRouter)

In [10]:
class DeepSeekCachingLLM:
    # Implicit caching via OpenRouter.
    # OpenRouter normalizes all provider responses to OpenAI format.
    # Cache stats are in prompt_tokens_details.cached_tokens — same field as OpenAI.
    # (NOT prompt_cache_read_tokens — that field doesn't exist on OpenRouter responses)

    _CHUNK_PROMPT = (
        'Here is the chunk to situate within the document above:\n'
        '<chunk>\n{chunk}\n</chunk>\n\n'
        'Give a short succinct context to situate this chunk for search retrieval. Nothing else.'
    )

    def situate_context(self, doc: str, chunk: str) -> Tuple[str, dict]:
        response = openrouter_client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[
                {
                    'role': 'system',                       # <- STATIC prefix (auto-cached >=64 tokens)
                    'content': f'<document>\n{doc}\n</document>',
                },
                {
                    'role': 'user',                         # <- DYNAMIC suffix
                    'content': self._CHUNK_PROMPT.format(chunk=chunk),
                },
            ],
        )
        # OpenRouter normalizes to OpenAI format: cache stats in prompt_tokens_details
        details  = getattr(response.usage, 'prompt_tokens_details', None)
        cached   = getattr(details, 'cached_tokens',     0) or 0
        cc_write = getattr(details, 'cache_write_tokens', 0) or 0
        # OpenRouter also exposes the actual cost — use it for accurate cost tracking
        actual_cost = getattr(response.usage, 'cost', None)
        return response.choices[0].message.content, {
            'input':          response.usage.prompt_tokens,
            'output':         response.usage.completion_tokens,
            'cache_creation': cc_write,
            'cache_read':     cached,
            'actual_cost':    actual_cost,  # OpenRouter-specific field (USD)
        }

deepseek_llm = DeepSeekCachingLLM()
print('DeepSeekCachingLLM ready')

DeepSeekCachingLLM ready


In [11]:
if not documents:
    print('No documents.')
else:
    print(f"{'Call':<6} {'total_input':>12} {'cache_read':>12} {'actual_cost':>13}  Status")
    print('-' * 58)

    deepseek_totals   = {'input': 0, 'output': 0, 'cache_creation': 0, 'cache_read': 0}
    deepseek_contexts = []
    total_cost        = 0.0

    for i, chunk in enumerate(demo_chunks, 1):
        ctx, usage = deepseek_llm.situate_context(first_doc['content'], chunk['content'])
        deepseek_contexts.append(ctx)
        for k in deepseek_totals:
            deepseek_totals[k] += usage[k]
        cost = usage.get('actual_cost') or 0
        total_cost += cost
        status = 'HIT' if usage['cache_read'] > 0 else 'MISS'
        cost_str = f'${cost:.6f}' if cost else 'N/A'
        print(f"  {i:<4} {usage['input']:>12,} {usage['cache_read']:>12,} {cost_str:>13}  {status}")

    print(f"\nTotals:")
    for k, v in deepseek_totals.items():
        print(f"   {k:<20} {v:>10,}")
    print(f"   {'actual_cost':<20} ${total_cost:.6f}")
    print()
    print("Note: OpenRouter normalizes DeepSeek responses to OpenAI format.")
    print("If cached_tokens=0 but actual_cost decreases across calls, caching IS active")
    print("at the upstream level — OpenRouter just may not surface the stats.")

Call    total_input   cache_read   actual_cost  Status
----------------------------------------------------------
  1           2,047            0     $0.000843  MISS
  2           2,117            0     $0.000714  MISS
  3           2,100            0     $0.000957  MISS
  4           2,100            0     $0.000733  MISS
  5           2,086            0     $0.000909  MISS

Totals:
   input                    10,450
   output                      276
   cache_creation                0
   cache_read                    0
   actual_cost          $0.004156

Note: OpenRouter normalizes DeepSeek responses to OpenAI format.
If cached_tokens=0 but actual_cost decreases across calls, caching IS active
at the upstream level — OpenRouter just may not surface the stats.


In [ ]:
p  = PRICING['deepseek']
t  = deepseek_totals
n  = len(demo_chunks)

cost_with    = ((t['input'] - t['cache_read']) * p['input'] / 1_000_000
                + t['cache_read'] * p['cache_read'] / 1_000_000
                + t['output'] * p['output'] / 1_000_000)
cost_without = t['input'] * p['input'] / 1_000_000 + t['output'] * p['output'] / 1_000_000
savings      = (1 - cost_with / cost_without) * 100 if cost_without > 0 else 0

print(f'DeepSeek Cost Analysis ({n} chunks)')
print(f'  Cost WITH caching:    ${cost_with:.6f}')
print(f'  Cost WITHOUT caching: ${cost_without:.6f}')
print(f'  Savings:              {savings:.1f}%')
scale = total_chunks / n
print(f"\nExtrapolated to {total_chunks} chunks:")
print(f'  WITH:    ${cost_with * scale:.4f}')
print(f'  WITHOUT: ${cost_without * scale:.4f}')

## Summary - Key Takeaways

1. **Prompt caching reuses the KV-cache** for repeated prefix tokens - 50-90% cheaper, 2-10x faster.

2. **Golden Rule**: static content FIRST, dynamic content LAST. Wrong order = no cache hits.

3. **Anthropic**: explicit opt-in via `cache_control: {type: ephemeral}` (native SDK only).
   90% off reads, 5-min TTL.

4. **OpenAI**: implicit, automatic, >=1024 tokens prefix. 50% off reads.
   Track: `response.usage.prompt_tokens_details.cached_tokens`

5. **DeepSeek via OpenRouter**: implicit, automatic, >=64 tokens. 90% off reads. Cheapest.
   Track: `getattr(response.usage, 'prompt_cache_read_tokens', 0)`

6. **In RAG**: N chunks per document = 1 miss + (N-1) hits = ~85% total LLM cost reduction.

---

### Provider Decision

| Priority | Use |
|----------|-----|
| Lowest cost | DeepSeek via OpenRouter (90% off, $0.14/1M) |
| Explicit control | Anthropic (cache_control, 90% off, 5-min TTL) |
| OpenAI ecosystem | GPT-4o-mini (automatic, 50% off, >=1024 tokens) |

### Token Field Reference

| Provider | Write Field | Read Field |
|----------|-------------|------------|
| Anthropic | `usage.cache_creation.ephemeral_5m_input_tokens` | `usage.cache_read_input_tokens` |
| OpenAI | not exposed | `usage.prompt_tokens_details.cached_tokens` |
| DeepSeek | `usage.prompt_cache_write_tokens` | `usage.prompt_cache_read_tokens` |